In [2]:
import numpy as np
import torch

In [3]:
def make_quantile_grid(M, device=None, dtype=torch.float32):
    """
    Inputs:
        M: int
    Outputs:
        taus: torch.Tensor shape (M,)
    """
    j = torch.arange(1, M + 1, device=device, dtype=dtype)
    taus = (j - 0.5) / M
    return taus

In [4]:
def huber_loss(u, k):
    """
    Inputs:
        u: torch.Tensor
        k: float
    Outputs:
        L: torch.Tensor, same shape as u
    """
    abs_u = torch.abs(u)
    quad = 0.5 * u * u
    lin = k * (abs_u - 0.5 * k)
    return torch.where(abs_u <= k, quad, lin)

In [5]:
def quantile_huber_loss(y, z, taus, k):
    """
    Critic QR loss (quantile Huber) du papier.

    Inputs:
        y: torch.Tensor (B, M) targets quantiles
        z: torch.Tensor (B, M) predicted quantiles
        taus: torch.Tensor (M,)
        k: float
    Outputs:
        loss: torch.Tensor scalar
    """
    # u_{i,j,j'} = y_{i,j'} - z_{i,j}
    # shapes: (B, M, 1) and (B, 1, M) -> (B, M, M)
    u = y.unsqueeze(1) - z.unsqueeze(2)  # (B, M, M)

    # indicator 1_{u < 0}
    indicator = (u < 0).to(u.dtype)      # (B, M, M)

    # taus for predicted quantiles dimension j : (1, M, 1)
    taus = taus.view(1, -1, 1).to(u.device).to(u.dtype)

    # weight = |tau - 1_{u<0}|
    weight = torch.abs(taus - indicator)  # (B, M, M)

    # huber on u
    L = huber_loss(u, k)                  # (B, M, M)

    # quantile huber loss
    loss = (weight * L).mean()
    return loss

In [6]:
def var_from_quantiles(quantiles, alpha=0.95):
    """
    Inputs:
        quantiles: torch.Tensor (B,M) or (M,)
        alpha: float
    Outputs:
        var: torch.Tensor (B,) or scalar tensor
    """
    q = quantiles
    if q.dim() == 1:
        q = q.unsqueeze(0)  # (1, M)

    B, M = q.shape
    idx = int(np.ceil(alpha * M) - 1)
    idx = max(0, min(M - 1, idx))

    var = q[:, idx]
    return var.squeeze(0) if quantiles.dim() == 1 else var

In [7]:
def cvar_from_quantiles(quantiles, alpha=0.95):
    """
    Inputs:
        quantiles: torch.Tensor (B,M) or (M,)
        alpha: float
    Outputs:
        cvar: torch.Tensor (B,) or scalar tensor
    """
    q = quantiles
    if q.dim() == 1:
        q = q.unsqueeze(0)

    B, M = q.shape
    idx = int(np.ceil(alpha * M) - 1)
    idx = max(0, min(M - 1, idx))

    tail = q[:, idx:]  # (B, M-idx)
    cvar = tail.mean(dim=1)
    return cvar.squeeze(0) if quantiles.dim() == 1 else cvar

In [8]:
def mean_std_objective_from_quantiles(quantiles, lambda_std=1.645):
    """
    Inputs:
        quantiles: torch.Tensor (B,M)
        lambda_std: float
    Outputs:
        obj: torch.Tensor (B,)
    """
    if quantiles.dim() == 1:
        quantiles = quantiles.unsqueeze(0)

    mean = quantiles.mean(dim=1)
    var = (quantiles - mean.unsqueeze(1)).pow(2).mean(dim=1)
    std = torch.sqrt(var + 1e-12)
    obj = mean + lambda_std * std
    return obj.squeeze(0) if obj.shape[0] == 1 else obj

In [9]:
def objective_from_quantiles(quantiles, objective, alpha=0.95, lambda_std=1.645):
    """
    Inputs:
        quantiles: torch.Tensor (B,M) or (M,)
        objective: str in {"mean_std","var","cvar"}
        alpha, lambda_std
    Outputs:
        obj: torch.Tensor (B,) or scalar
    """
    obj = objective.lower()
    if obj == "mean_std":
        return mean_std_objective_from_quantiles(quantiles, lambda_std=lambda_std)
    elif obj == "var":
        return var_from_quantiles(quantiles, alpha=alpha)
    elif obj == "cvar":
        return cvar_from_quantiles(quantiles, alpha=alpha)
    else:
        raise ValueError(f"Unknown objective={objective}. Choose from {{'mean_std','var','cvar'}}.")

In [ ]:
RUN_TEST = False

if RUN_TEST:
    torch.manual_seed(0)

    B = 4
    M = 100
    taus = make_quantile_grid(M)

    # Fake "loss distribution quantiles": increasing with tau
    base = torch.linspace(-1.0, 2.0, M).unsqueeze(0).repeat(B, 1)
    noise = 0.1 * torch.randn(B, M)
    q = base + noise
    q, _ = torch.sort(q, dim=1)

    print("VaR(0.95):", var_from_quantiles(q, 0.95))
    print("CVaR(0.95):", cvar_from_quantiles(q, 0.95))
    print("Mean-Std:", mean_std_objective_from_quantiles(q, 1.645))

    # Test quantile huber loss (targets vs preds)
    z = q + 0.2 * torch.randn(B, M)  # predictions noisy
    y = q                             # targets
    loss = quantile_huber_loss(y, z, taus, k=1.0)
    print("Quantile Huber loss:", float(loss))

VaR(0.95): tensor([1.8518, 1.8892, 1.8277, 1.9329])
CVaR(0.95): tensor([1.9764, 1.9673, 1.9282, 2.0246])
Mean-Std: tensor([1.9682, 1.9551, 1.9420, 1.9915])
Quantile Huber loss: 0.13985957205295563
